# Generate Kaggle Submission
Run your best model configuration on `test.jsonl` to produce the submission CSV.

In [ ]:
# === IMPORT LIBRARIES === 
import sys
import os
from pathlib import Path
import ctypes

In [ ]:
# === RESOLVE PATHS ===

# CRITICAL SYSTEM BOOT FIX: Force-inject CUDA library to RAM before ANY module imports!
try:
    ctypes.CDLL("/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/nvidia/cu13/lib/libnvJitLink.so.13")
except Exception:
    pass

# SYSTEM HOTFIX: Inject absolute path to CUDA 13.0 linker libraries
# This guarantees that bitsandbytes and 4-bit quantization load flawlessly on this server!
cuda_link_path = "/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/nvidia/cu13/lib"
os.environ["LD_LIBRARY_PATH"] = os.environ.get("LD_LIBRARY_PATH", "") + ":" + cuda_link_path
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Add parent directory to python path to import from src/
sys.path.append(str(Path("..").resolve()))

In [ ]:
# === IMPORT MODULES ===
import pandas as pd
from tqdm import tqdm
from src.data.loader import load_jsonl
from src.retrievers.bm25 import BM25Retriever
from src.retrievers.dense import DenseRetriever
from src.retrievers.llm import LLMRetriever
from src.retrievers.cross_encoder import CrossEncoderRetriever
from src.evaluation.naming import generate_experiment_name

# === IMPORT HF API TOKEN ===
env_path = Path("..") / ".env"
if env_path.exists():
    with open(env_path, "r") as f:
        for line in f:
            if "=" in line and not line.strip().startswith("#"):
                k, v = line.strip().split("=", 1)
                os.environ[k.strip()] = v.strip()
    print("✅ Successfully Authenticated Hugging Face Token!")

In [ ]:
# === LOAD DATA ===
DATA_DIR = Path("../data")
test_data = load_jsonl(str(DATA_DIR / "test.jsonl"))
print(f"Loaded {len(test_data)} test samples.")

In [ ]:
# === SUBMISSION CONFIGURATION ===
CONFIG = {
    "retriever": "llm",
    "dense_model_name": "BAAI/bge-m3", # Ignored
    "llm_model_name": "Qwen/Qwen2.5-32B-Instruct",
    "cross_encoder_model_name": "BAAI/bge-reranker-v2-m3", # Ignored
    "dense_scores_path": "../outputs/cache/ensemble_dense_scores/qwen3-embedding-8b_w0.6_bge-m3_w0.4_dense_scores_test.json", # <--- IMPORTANT: must be test.json
    "top_k": 5,
    "max_candidates": 16, 
    "max_chars": 1000 # Adjust if you changed this during your best run
}
os.environ["DISABLE_VLM"] = "1"
os.environ["VLM_CAPTIONS_FILE"] = "vlm_image_captions/qwen2-vl-7b-instruct_image_captions.json" # Ignored since VLM is disabled

In [ ]:
# === RETRIEVER SELECTION ===
if CONFIG["retriever"] == "bm25":
    retriever = BM25Retriever()
elif CONFIG["retriever"] == "dense":
    retriever = DenseRetriever(
        model_name=CONFIG["dense_model_name"]
    )
elif CONFIG["retriever"] == "llm":
    retriever = LLMRetriever(
        model_name=CONFIG["llm_model_name"], 
        load_in_4bit=True, 
        disable_filtering=False,
        dense_scores_path=CONFIG["dense_scores_path"],
        max_candidates=CONFIG.get("max_candidates", 12),
        max_chars=CONFIG.get("max_chars", None)
    )
elif CONFIG["retriever"] == "cross_encoder":
    retriever = CrossEncoderRetriever(
        model_name=CONFIG["llm_model_name"],
        load_in_4bit=True,
        disable_filtering=False,
        dense_scores_path=CONFIG["dense_scores_path"],
        max_candidates=CONFIG.get("max_candidates", 12),
        max_chars=CONFIG.get("max_chars", None)
    )
else:
    raise NotImplementedError(f"Retriever type '{CONFIG['retriever']}' not recognized.")

In [ ]:
# === RUN PREDICTION ===
predictions = []
for sample in tqdm(test_data, desc="Generating predictions"):
    preds = retriever.retrieve(sample, top_k=5)
    pred_ids = [p[0] for p in preds]
    predictions.append({
        "q_id": sample.q_id,
        "gold_quotes": " ".join(pred_ids)
    })

In [ ]:
# === SAVE SUBMISSION ===
from pathlib import Path
from src.evaluation.naming import generate_experiment_name
output_dir = Path("../outputs/submissions")
output_dir.mkdir(parents=True, exist_ok=True)
df_sub = pd.DataFrame(predictions)

# Generate name dynamically from CONFIG and Environment Variables!
exp_name = generate_experiment_name(CONFIG)
sub_name = f"submission_{exp_name}.csv" 
output_path = output_dir / sub_name
df_sub.to_csv(output_path, index=False)
print(f"\n✅ Successfully saved predictions to: {output_path.name}")